In [2]:
# we use pandas for handling our dataframes / tables
import pandas as pd

# Prophet is a library from Facebook for time series prediction
from prophet import Prophet

# yfinance accesses yahoo finance to pull stock data
import yfinance as yf


In [3]:
# we define our stockname here, change it here to apply to a different stock, eg. to "MSFT"
# this will also be used to dynamically label our output
stockname = "AAPL"

# this gives us a timestamp
import datetime as dt
# we will also use this later to dynamically name our output files
time = dt.datetime.today().strftime("%Y%m%d")

# we call yahoo finance to get information about our stock
stock = yf.Ticker(stockname)

# get stock info
stock.info

{'address1': 'One Apple Park Way',
 'city': 'Cupertino',
 'state': 'CA',
 'zip': '95014',
 'country': 'United States',
 'phone': '(408) 996-1010',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'industryKey': 'consumer-electronics',
 'industryDisp': 'Consumer Electronics',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download app

In [4]:
# get historical market data
hist = stock.history(period="max", auto_adjust=True)

# show the head of the dataframe we just downloaded
# we do this by using head() on the dataframe.
# This also shows us the earliest data points
hist.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
1980-12-12 00:00:00-05:00,0.098297,0.098725,0.098297,0.098297,469033600,0.0,0.0
1980-12-15 00:00:00-05:00,0.093597,0.093597,0.093169,0.093169,175884800,0.0,0.0
1980-12-16 00:00:00-05:00,0.086758,0.086758,0.086331,0.086331,105728000,0.0,0.0
1980-12-17 00:00:00-05:00,0.088468,0.088895,0.088468,0.088468,86441600,0.0,0.0
1980-12-18 00:00:00-05:00,0.091032,0.091460,0.091032,0.091032,73449600,0.0,0.0


In [11]:
# this shows us some quick statistics
hist.describe()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
count,10923.000000,10923.000000,10923.000000,10923.000000,1.092300e+04,10923.000000,10923.000000
mean,20.313329,20.532631,20.102713,20.327356,3.198115e+08,0.000735,0.001556
std,43.309119,43.774638,42.872624,43.346819,3.359362e+08,0.011550,0.083950
min,0.038383,0.038383,0.037952,0.037952,0.000000e+00,0.000000,0.000000
25%,0.241472,0.246234,0.236640,0.241885,1.144586e+08,0.000000,0.000000
50%,0.422750,0.431087,0.416786,0.424065,2.077040e+08,0.000000,0.000000
75%,16.892520,17.031843,16.699226,16.879339,3.998652e+08,0.000000,0.000000
max,197.767647,199.365599,196.748941,197.857529,7.421641e+09,0.240000,7.000000


In [5]:
# we want out input table to have 2 columns, ds and y, so let’s just copy the historical dates (hist.index) and adjusted closing prices (hist[‘Close’]) to a new DataFrame.

# we're creating a new DataFrame
df = pd.DataFrame()

# We create a column called "ds" in the new data frame and fill it with the index from hist (this is the data)
df['ds'] = hist.index.values

# We also create a column called y, this is filled with the Close value from hist
df['y'] = hist['Close'].values

# just as we used heads before to look at the beginning of a Dataframe, we can use tail to look at the last values
df.tail()

,ds,y
11382,2026-02-11 05:00:00,275.500000
11383,2026-02-12 05:00:00,261.730011
11384,2026-02-13 05:00:00,255.779999
11385,2026-02-17 05:00:00,263.880005
11386,2026-02-18 05:00:00,264.350006


In [6]:
# This calls Prophet
m = Prophet( )

# we are now using Prophet to learn past time series
m.fit(df)

10:10:08 - cmdstanpy - INFO - Chain [1] start processing
10:10:14 - cmdstanpy - INFO - Chain [1] done processing


In [7]:
# lets "create the future" aka create a dataframe for it
# we want to forecast 365 periods into the future, prohpet automatically understands that our dates are days
future = m.make_future_dataframe(periods=1)

# right now our dataframe is empty and just shows future dates. let's see if it counted correctly, by looking at the last values
future.tail()

,ds
11383,2026-02-12 05:00:00
11384,2026-02-13 05:00:00
11385,2026-02-17 05:00:00
11386,2026-02-18 05:00:00
11387,2026-02-19 05:00:00


In [8]:
# now we use predict to fill our new dataframe with the values
forecast = m.predict(future)

# we also filter the columns along the way, because Prophet gives you a lot more output to work with
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

,ds,yhat,yhat_lower,yhat_upper
11383,2026-02-12 05:00:00,245.870407,235.855397,256.379589
11384,2026-02-13 05:00:00,245.984926,236.475187,256.418891
11385,2026-02-17 05:00:00,245.989874,235.522012,256.227001
11386,2026-02-18 05:00:00,246.053199,236.598800,256.876740
11387,2026-02-19 05:00:00,245.900374,236.305603,256.811889


In [9]:
#pull last item from historic data
last_market_value = df["y"].iloc[-1]

#print last market value
#print (last_market_value)

In [10]:
print(last_market_value)

264.3500061035156


In [11]:
#pull last item from forecast data
last_forecast_value = forecast["yhat"].iloc[-1]

#print last forecast value
#print (last_forecast_value)

In [12]:
# Lets create a simple check to compare the two values
# we just compare the two values we calculated and return the
#corresponding statement

if last_market_value > last_forecast_value:
    print ("Overrated")
elif last_market_value == last_forecast_value:
    print ("Fairly valued")
else:
    print ("Underrated")

Overrated
